# Testing `atmosphere_characterization_tools.py`

Validates each of the 7 standalone functions in
[`aott/atmosphere_characterization_tools.py`](../aott/atmosphere_characterization_tools.py)
against a simulated, closed-loop-only observation file `test.hdf5`.

The original `Atmosphere_Characterization` class is **not** treated as ground
truth here: section 4 found that its `temporal_structure_function` has a
`diameter**2` scaling bug (see the module docstring in
`atmosphere_characterization_tools.py` for the proof), so its `tau0`/`V0`
numbers are known to be wrong. The class is only used, narrowly, as a
formula-fidelity regression check for the two functions this rewrite did
*not* change the math of (`estimate_r0_L0`, `estimate_wind_gain_delay_from_psd`)
-- i.e. "does the rewrite reproduce the old formula," not "is the old formula
correct."

See `papers/Literature_Comparison.md` and `class reports/Atmosphere_Characterization.md`
for the background these functions and this comparison are based on.

In [ ]:
import importlib.util
import numpy as np
import h5py
import matplotlib.pyplot as plt

%matplotlib inline

REPO = r"C:\Users\foyarzun\Nextcloud\AOTelemetryToolbox"
FILE_NAME = REPO + r"\simulation\test.hdf5"


def load_module(name, path):
    # aott/__init__.py imports PSF_Processing, which needs maoppy (not
    # installed on this machine per CLAUDE.md) -- load modules directly by
    # path instead of going through the aott package.
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


act = load_module("atmosphere_characterization_tools", REPO + r"\aott\atmosphere_characterization_tools.py")
acz = load_module("Atmosphere_Characterization", REPO + r"\aott\Atmosphere_Characterization.py")

In [ ]:
# 1. Inspect the file

with h5py.File(FILE_NAME, "r") as f:
    D = float(f["Calibration"].attrs["Diameter"])
    wavelength = float(f["Calibration"].attrs["AO_Calibration_Wavelength"])
    leak = float(f["WFS"].attrs["Loop_Leak"])
    gain = float(f["WFS"].attrs["Loop_Gain"])
    freq = float(f["WFS"].attrs["Loop_Freq"])
    loop_status = f["WFS"].attrs["loop_status"][:]
    dm_commands = f["WFS"]["DM_commands"][:]
    dm_timestamps = f["WFS"]["DM_TimeStamps"][:]
    C2Z = f["Calibration"]["C2Z"][:]

print(f"Diameter = {D} m, AO_Calibration_Wavelength = {wavelength*1e9:.0f} nm")
print(f"Loop_Freq = {freq} Hz, Loop_Gain = {gain}, Loop_Leak = {leak}")
print(f"DM_commands: {dm_commands.shape}, C2Z: {C2Z.shape} (-> {C2Z.shape[0]} Zernike modes)")
print(f"loop_status: {loop_status.shape}, closed-loop fraction = {np.mean(loop_status != 0):.2f}")

## 2. Build the Zernike-mode input -- two gotchas found while writing this notebook

`atmosphere_characterization_tools.py`'s interface convention says `zernike_modes`
must already be in the same physically-calibrated radians `dm_commands @ C2Z.T`
produces in the existing pipeline, and `timestamps` should be real per-sample
times. Two things about *this specific file* had to be gotten right before that
was true:

1. **Actuator-mean removal.** `Atmosphere_Characterization.LoadData` subtracts
   each timestep's mean DM command *across actuators* (`axis=1`) before
   projecting to Zernike modes -- a piston-like offset. Skipping this step
   before calling `dm_commands @ C2Z.T` gives a *different* result, barely
   visible in `r0`/`L0` but large enough to change which local optimum the
   PSD-based gain/delay fit converges to. Do this before feeding real
   DM-command telemetry into these tools.
2. **`DM_TimeStamps` doesn't reflect the true loop rate in this file.** The
   median spacing between recorded `DM_TimeStamps` is tens of milliseconds,
   not the 1 ms implied by `Loop_Freq = 1000` Hz -- these look like
   wall-clock timestamps from however `simulation/DataGeneration.ipynb`
   generated this file, not real per-frame AO-loop timestamps. Every function
   here that derives a sample period from `timestamps`
   (`np.median(np.diff(timestamps))`) will silently use the wrong rate if fed
   `DM_TimeStamps` directly on this file. The cross-checks below use a
   nominal, evenly-spaced timestamp array built from `Loop_Freq` instead. On
   *real* telemetry the recorded per-frame timestamps should be trustworthy
   and are the right thing to pass in directly -- check this assumption
   against your own data before trusting it there too.

In [ ]:
print("DM_TimeStamps median diff:", np.median(np.diff(dm_timestamps)), "s   vs 1/Loop_Freq:", 1 / freq, "s")

dm_commands_centered = dm_commands - dm_commands.mean(axis=1, keepdims=True)
dm_Z_modes = dm_commands_centered @ C2Z.T
print("dm_Z_modes:", dm_Z_modes.shape)

In [ ]:
# 3. Pick a closed-loop batch -- this file has alternating open/closed-loop segments

changes = np.where(np.diff(loop_status) != 0)[0] + 1
run_starts = np.concatenate(([0], changes))
run_ends = np.concatenate((changes, [len(loop_status)]))
run_status = loop_status[run_starts]

print("segments (start, end, status):")
for s, e, st in zip(run_starts, run_ends, run_status):
    print(f"  [{s:4d}:{e:4d}]  length {e-s:4d}  status {st}")

closed_runs = [(s, e) for s, e, st in zip(run_starts, run_ends, run_status) if st != 0]
batch_start, batch_end = max(closed_runs, key=lambda se: se[1] - se[0])
n_batch = batch_end - batch_start
print(f"\nusing the largest closed-loop run: [{batch_start}:{batch_end}], length {n_batch}")

zernike_batch = dm_Z_modes[batch_start:batch_end]
timestamps_batch = np.arange(n_batch) / freq  # nominal, evenly spaced -- see the gotcha above

## 4. Formula-fidelity regression check (not a correctness baseline)

Runs the *unmodified* class on the exact same batch, but only calls
`ComputeR0` and `ZernikeWindEstimation` -- not `ComputeTau0`/`ComputeV0`,
whose `temporal_structure_function` has the confirmed `diameter**2` bug (see
the module docstring and the check in section 6). This only tells us whether
`estimate_r0_L0`/`estimate_wind_gain_delay_from_psd` reproduce the old
formulas exactly; it says nothing about whether those formulas are
themselves correct.

In [ ]:
atm = acz.Atmosphere_Characterization(FILE_NAME, batch_size=n_batch, filter_TT=False)
atm.batch_start = batch_start
atm.LoadData()
atm.ComputeR0(display=False)          # ok to trust -- unchanged math, see section 5
atm.ZernikeWindEstimation()           # ok to trust -- unchanged math, see section 7
# NOT calling atm.ComputeTau0()/atm.ComputeV0(): temporal_structure_function has the
# confirmed diameter**2 bug (section 6), so those numbers are known wrong -- don't use them.

assert np.allclose(atm.dm_Z_modes, zernike_batch), "sanity check: our manual slice must match the class's own dm_Z_modes"

regression = dict(
    r0_cm=atm.r0 * 100, L0_m=atm.L0,
    V0_Zernike=atm.zernike_wind_speed, gain=atm.effective_gain, delay=atm.effective_delay,
)
for k, v in regression.items():
    print(f"{k:12s} = {v:.4f}")

In [ ]:
# 5. estimate_r0_L0

r0l0 = act.estimate_r0_L0(zernike_batch, D)
print(f"r0 = {r0l0.r0*100:.4f} cm  (old-formula regression check: {regression['r0_cm']:.4f} cm)")
print(f"L0 = {r0l0.L0:.4f} m    (old-formula regression check: {regression['L0_m']:.4f} m)")

fig, ax = plt.subplots()
ax.semilogy(r0l0.radial_orders, r0l0.measured_variance, "o", label="measured")
ax.semilogy(r0l0.radial_orders, r0l0.model_variance, "-", label="von Karman fit")
ax.set_xlabel("radial order n")
ax.set_ylabel("Zernike coefficient variance (rad$^2$)")
ax.legend()
ax.set_title("estimate_r0_L0")
plt.show()

## 6. `estimate_tau0_v0_structure_function` -- the old `diameter**2` factor is a confirmed bug

`Atmosphere_Characterization.temporal_structure_function` scales its result by
`diameter ** 2`. The fit model it's compared against,
`PhaseTemporalStructureFunction(delay, wind_speed, r0) = 6.88*(V*tau/r0)**(5/3)`,
is the standard Kolmogorov point-to-point phase structure function (Conan
2008 eq. 17-19) -- it depends only on the separation `V*tau`, never on the
pupil diameter. For the measured curve to be fit against that model
consistently, it has to be the same kind of quantity.

The cell below checks this directly on real data, three ways: the old
`DM_modes`-based phase map (i) with the old `* diameter**2` as committed, and
(ii) with it removed, plus (iii) this module's independent `C2Z`-based
Zernike version (which never had the factor). (ii) and (iii) use two
different, independently-fit calibration matrices, so they won't match to
machine precision -- but they should agree to a few percent if both are
correctly scaled, and (i) should differ from them by a clean factor of
`diameter**2` if that term is simply extra.

In [ ]:
with h5py.File(FILE_NAME, "r") as f:
    dm_modes = f["Calibration"]["DM_modes"][:]

dm_batch_centered = dm_commands[batch_start:batch_end] - dm_commands[batch_start:batch_end].mean(axis=1, keepdims=True)

full_dm_map = np.tensordot(dm_batch_centered, dm_modes, axes=(1, 0))
full_dm_map -= full_dm_map.mean(axis=0, keepdims=True)
full_dm_map *= 2 * np.pi / wavelength

max_lag_check = 20
D_tau_old_with_D2 = acz.temporal_structure_function(full_dm_map, D, max_lag_check)
D_tau_old_no_D2 = acz.temporal_structure_function(full_dm_map, 1.0, max_lag_check)  # diameter=1 -> no-op scaling
_, D_tau_zernike = act.zernike_structure_function(zernike_batch, timestamps_batch, max_lag_check)

print(f"{'lag':>5}  {'old, WITH *D^2':>16}  {'old, NO *D^2':>14}  {'Zernike/C2Z, no *D^2':>22}  {'ratio (WITH / no)':>18}")
for i in range(0, max_lag_check + 1, 4):
    ratio = D_tau_old_with_D2[i] / D_tau_old_no_D2[i] if D_tau_old_no_D2[i] else float("nan")
    print(f"{i:5d}  {D_tau_old_with_D2[i]:16.4f}  {D_tau_old_no_D2[i]:14.4f}  {D_tau_zernike[i]:22.4f}  {ratio:18.3f}")
print(f"\nD^2 = {D**2}")

In [ ]:
tau0v0 = act.estimate_tau0_v0_structure_function(zernike_batch, timestamps_batch, r0l0.r0)
print(f"tau0 = {tau0v0.tau0*1000:.4f} ms")
print(f"V0   = {tau0v0.V0:.4f} m/s")
print(f"crossed 1 rad^2 threshold: {tau0v0.crossed_threshold}, at lag index {tau0v0.max_lag_reached}")

fig, ax = plt.subplots()
ax.plot(tau0v0.lags * 1000, tau0v0.structure_function, "o-")
ax.axhline(1.0, color="k", linestyle=":", label="1 rad$^2$ crossing")
ax.axvline(tau0v0.tau0 * 1000, color="r", linestyle="--", label=f"tau0 = {tau0v0.tau0*1000:.2f} ms")
ax.set_xlabel("lag (ms)")
ax.set_ylabel("structure function D(tau)  (rad$^2$)")
ax.legend()
ax.set_title("estimate_tau0_v0_structure_function")
plt.show()

In [ ]:
# 7. estimate_wind_gain_delay_from_psd

# nperseg=n_batch matches what the old class's hard-coded GetSignalPSD(nperseg=1000)
# effectively does here (scipy silently caps nperseg at the input length), for a
# tight regression check; nperseg=256 below gives genuine multi-segment Welch
# averaging instead, which is what you'd actually want to use in practice.
windgd = act.estimate_wind_gain_delay_from_psd(zernike_batch, timestamps_batch, D, leak, nperseg=n_batch, max_radial_order=6)
print("-- max_radial_order=6, nperseg=n_batch (matches regression check's mode count and effective nperseg) --")
print(f"V0     = {windgd.V0:.4f} m/s   (old-formula regression check, V0_Zernike: {regression['V0_Zernike']:.4f})")
print(f"gain   = {windgd.effective_gain:.4f}     (regression check: {regression['gain']:.4f})")
print(f"delay  = {windgd.effective_delay:.4f}     (regression check: {regression['delay']:.4f})")

# using all 50 modes the calibration actually provides, and a realistic multi-segment
# nperseg -- the original class never does either (nModes is hard-coded to 27, and
# nperseg=1000 is hard-coded regardless of batch length)
nperseg = min(256, n_batch)
windgd_all = act.estimate_wind_gain_delay_from_psd(zernike_batch, timestamps_batch, D, leak, nperseg=nperseg)
print(f"\n-- all 50 modes (orders 2-9), nperseg={nperseg}, beyond what the original class ever fits --")
print(f"V0 = {windgd_all.V0:.4f} m/s, gain = {windgd_all.effective_gain:.4f}, delay = {windgd_all.effective_delay:.4f}")

fig, ax = plt.subplots()
cmap = plt.get_cmap("viridis")
colors = cmap(np.linspace(0, 1, len(windgd.radial_orders)))
for i, n in enumerate(windgd.radial_orders):
    ax.loglog(windgd.frequency, windgd.psd[i], color=colors[i], label=f"n={n}")
    ax.loglog(windgd.frequency, windgd.fitted_psd[i], color=colors[i], linestyle="--")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("PSD")
ax.legend(fontsize=8)
ax.set_title("estimate_wind_gain_delay_from_psd -- measured (solid) vs fitted (dashed)")
plt.show()

In [ ]:
# 8. estimate_wfs_noise_variance
# this simulated file appears to have little/no injected WFS noise, so
# near-zero noise variance here is the expected result, not a broken output.

noise = act.estimate_wfs_noise_variance(zernike_batch, timestamps_batch)
fig, ax = plt.subplots()
ax.bar(np.arange(len(noise.noise_variance)), noise.noise_variance)
ax.set_xlabel("mode index (column of zernike_batch)")
ax.set_ylabel("estimated noise variance (rad$^2$)")
ax.set_title("estimate_wfs_noise_variance")
plt.show()
print("max noise variance:", noise.noise_variance.max())

## 9. `estimate_wind_speed_autocorrelation_cutoff` -- implausible, and pseudo-open-loop input does NOT fix it

Run directly on `zernike_batch` (raw closed-loop DM commands), this gives an
implausible wind speed. The natural guess -- try it with
`reconstruct_pseudo_open_loop` input instead -- turns out **not** to be the
fix. This file's `WFS_measurements` is a single frame, not a time series, so
that can't be tested on real data here; testing it required a controlled
synthetic experiment (known wind speed in, check what comes out), and that
experiment showed the function is wrong even on a *known-true, pure
open-loop* synthetic atmosphere, before any loop or reconstruction enters the
picture at all.

The actual cause, confirmed analytically (cell below, zero noise, zero
sampling involved): NAOS/Fusco et al. 2004's `1.15*pi` constant (eq. 9) was
calibrated against Conan et al. 1995's *exact* theoretical Zernike temporal
PSD shape. This codebase's own atmosphere model elsewhere
(`_low_pass(f, f_c, 17/3)`, used by `estimate_wind_gain_delay_from_psd`) is
only an approximation of that shape -- and applying `1.15*pi/tau_1e` to the
*exact* autocorrelation implied by this simplified shape recovers a cutoff
frequency that is consistently **~11.7x too high**, for any true `f_c`. That
alone explains most of the implausible result; see
`atmosphere_characterization_tools.py`'s docstring for
`estimate_wind_speed_autocorrelation_cutoff` for the full writeup. The
AO-loop-bandwidth correlation shown below is still real, but it's a smaller,
secondary effect riding on top of this larger, more fundamental one -- not
the primary explanation this section originally gave.

In [ ]:
autoc = act.estimate_wind_speed_autocorrelation_cutoff(zernike_batch, timestamps_batch, D, r0=r0l0.r0)
print(f"V0 = {autoc.V0:.1f} m/s (implausible), tau0 = {autoc.tau0*1000:.4f} ms")
print("per-order autocorrelation cutoff frequency (Hz):", autoc.cutoff_frequency)
print("for comparison, the TRUE atmosphere knee from the open-loop-compensated PSD fit (Hz):")
print(windgd.per_order_params["cutoff_frequency"])

# where does the AO loop's OWN bandwidth sit, using the SAME fitted gain/delay?
f_scan = np.logspace(-1, 2.9, 2000)
H_hold, H_delay, H_controller = act._ao_loop_transfer_functions(f_scan, 1 / freq, windgd.effective_gain, leak, windgd.effective_delay)
loop = H_hold * H_delay * H_controller
command_tf = np.abs(loop / (1 + H_hold * loop)) ** 2  # atmosphere -> DM-command transfer function
loop_bandwidth_hz = f_scan[np.where(command_tf < 0.5)[0][0]]
print(f"\nAO loop's own bandwidth (command_tf drops below 0.5, from gain={windgd.effective_gain:.3f}, delay={windgd.effective_delay:.3f}): {loop_bandwidth_hz:.1f} Hz")
print("-> this, not the ~1-5 Hz atmosphere knee, is what the autocorrelation cutoff frequencies above are close to.")

fig, ax = plt.subplots()
ax.semilogx(f_scan, command_tf)
ax.axhline(0.5, color="k", linestyle=":")
ax.axvline(loop_bandwidth_hz, color="r", linestyle="--", label=f"loop bandwidth = {loop_bandwidth_hz:.0f} Hz")
for fc in autoc.cutoff_frequency:
    ax.axvline(fc, color="gray", alpha=0.3)
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("|atmosphere -> DM command|$^2$")
ax.legend()
ax.set_title("AO loop's own transfer function (gray lines: autocorrelation cutoff frequencies per order)")
plt.show()

In [ ]:
# Analytical check: apply the 1.15*pi/tau_1e formula to the EXACT autocorrelation
# implied by this codebase's own atmosphere PSD shape (_low_pass(f, f_c, 17/3)),
# computed by direct numerical integration -- no synthetic signal, no noise, no
# sampling at all. If the constant were right for this shape, the ratio below
# would be 1.0 for every f_c.

from scipy.interpolate import interp1d as _interp1d


def _analytical_1e_width_ratio(f_c_true, alpha1=17 / 3, f_max=2000.0, n_f=400_000):
    f = np.linspace(0, f_max, n_f)
    psd = act._low_pass(f, f_c_true, alpha1)
    df = f[1] - f[0]
    taus = np.linspace(0, 5.0 / f_c_true, 2000)
    R = np.array([np.sum(psd * np.cos(2 * np.pi * f * t)) * df for t in taus])
    R_norm = R / R[0]
    i1 = np.where(R_norm <= 1 / np.e)[0][0]
    tau_1e = float(_interp1d(R_norm[i1 - 1:i1 + 1], taus[i1 - 1:i1 + 1])(1 / np.e))
    return (1.15 * np.pi / tau_1e) / f_c_true


print("true f_c (Hz)   implied/true ratio   (should be 1.0 if the 1.15*pi constant matched this shape)")
for f_c_true in [1.0, 3.0, 5.0, 10.0]:
    ratio = _analytical_1e_width_ratio(f_c_true)
    print(f"{f_c_true:12.1f}   {ratio:20.3f}")

In [ ]:
# Controlled experiment: does reconstruct_pseudo_open_loop fix the wind-speed
# estimate? Needs a KNOWN true wind speed and real open-loop WFS data to check
# against -- this file has neither (WFS_measurements is a single frame), so
# both are built synthetically: a Conan et al. 1995-shaped atmosphere (flat
# low frequencies, f^-17/3 high frequencies, cutoff f_c=0.3*(n+1)*V_true/D per
# radial order) driving a standard, stable, negative-feedback leaky-integrator
# loop with this file's real gain/leak.


def _colored_noise(freqs_target, psd_target, n_samples, fs, rng):
    freqs = np.fft.rfftfreq(n_samples, d=1 / fs)
    psd_vals = np.interp(freqs, freqs_target, psd_target, left=psd_target[0], right=psd_target[-1])
    psd_vals[0] = psd_vals[1]
    amplitude = np.sqrt(psd_vals * fs * n_samples / 2)
    phases = rng.uniform(0, 2 * np.pi, len(freqs))
    spectrum = amplitude * np.exp(1j * phases)
    spectrum[0] = amplitude[0]
    if n_samples % 2 == 0:
        spectrum[-1] = amplitude[-1]
    return np.fft.irfft(spectrum, n=n_samples)


rng = np.random.default_rng(2)
n_sim = 8000
n_sim_modes = 50
V_true = windgd_all.V0  # reuse the earlier PSD-fit's V0 as a plausible target, for comparability
sim_orders = act.zernike_radial_orders(n_sim_modes, first_noll_index=2)
f_model = np.linspace(0.01, freq / 2, 8000)

a_atm_sim = np.zeros((n_sim, n_sim_modes))
for j in range(n_sim_modes):
    n = sim_orders[j]
    if n < 2:
        continue
    f_c = 0.3 * (n + 1) * V_true / D
    psd_target = (1.0 / (n + 1) ** 3) * act._low_pass(f_model, f_c, 17 / 3)
    a_atm_sim[:, j] = _colored_noise(f_model, psd_target, n_sim, freq, rng)
timestamps_sim = np.arange(n_sim) / freq

sim_delay = 2
dm_sim = np.zeros((n_sim, n_sim_modes))
wfs_sim = np.zeros((n_sim, n_sim_modes))  # standard convention: wfs = atmosphere - DM_shape
for t in range(n_sim):
    dm_prev = dm_sim[t - 1] if t >= 1 else np.zeros(n_sim_modes)
    wfs_sim[t] = a_atm_sim[t] - dm_prev
    residual_delayed = wfs_sim[t - sim_delay] if t >= sim_delay else np.zeros(n_sim_modes)
    dm_sim[t] = leak * dm_prev + gain * residual_delayed  # standard negative-feedback leaky integrator
assert np.all(np.isfinite(dm_sim)), "closed-loop simulation diverged"

r0l0_sim = act.estimate_r0_L0(a_atm_sim, D)
autoc_true_atm = act.estimate_wind_speed_autocorrelation_cutoff(a_atm_sim, timestamps_sim, D, r0=r0l0_sim.r0)
autoc_closed_sim = act.estimate_wind_speed_autocorrelation_cutoff(dm_sim, timestamps_sim, D, r0=r0l0_sim.r0)
recon_sim, recon_ts_sim = act.reconstruct_pseudo_open_loop(dm_sim, wfs_sim, timestamps_sim, frame_delay=sim_delay)
autoc_recon_sim = act.estimate_wind_speed_autocorrelation_cutoff(recon_sim, recon_ts_sim, D, r0=r0l0_sim.r0)

print(f"true V0 (by construction)                    : {V_true:.2f} m/s")
print(f"estimated on the TRUE open-loop atmosphere    : {autoc_true_atm.V0:.2f} m/s")
print(f"estimated on raw closed-loop DM commands      : {autoc_closed_sim.V0:.2f} m/s")
print(f"estimated on reconstructed pseudo-open-loop   : {autoc_recon_sim.V0:.2f} m/s")
print("\n-> reconstruction does not fix it: even the TRUE atmosphere is off by roughly the same factor.")

In [ ]:
# 10. detect_secondary_layer_candidate

secondary = act.detect_secondary_layer_candidate(zernike_batch, timestamps_batch, D, leak, nperseg=nperseg, max_radial_order=6)
print("detected:", secondary.detected, " candidate_V0:", secondary.candidate_V0)

fig, ax = plt.subplots()
ax.bar([str(n) for n in secondary.radial_orders], secondary.residual_power_fraction)
ax.axhline(0.15, color="k", linestyle=":", label="detection threshold")
ax.set_xlabel("radial order n")
ax.set_ylabel("residual power fraction")
ax.legend()
ax.set_title("detect_secondary_layer_candidate")
plt.show()

In [ ]:
# 11. reconstruct_pseudo_open_loop -- synthetic demo
# this file's WFS/WFS_measurements is a single frame, not a (n_samples, n_actuators)
# time series aligned with DM_commands, so it can't be used here for a real
# reconstruction; a small synthetic perturbation stands in for it instead.

rng = np.random.default_rng(0)
synthetic_wfs = zernike_batch + rng.normal(0, 0.05 * zernike_batch.std(), zernike_batch.shape)
recon, recon_timestamps = act.reconstruct_pseudo_open_loop(zernike_batch, synthetic_wfs, timestamps_batch, frame_delay=2)
print("reconstructed shape:", recon.shape, " (", zernike_batch.shape[0], "->", recon.shape[0], "after the frame-delay trim)")

fig, ax = plt.subplots()
ax.plot(timestamps_batch * 1000, zernike_batch[:, 0], label="DM-derived (as recorded)")
ax.plot(recon_timestamps * 1000, recon[:, 0], label="reconstructed pseudo-open-loop")
ax.set_xlabel("time (ms)")
ax.set_ylabel("mode 0 coefficient (rad)")
ax.legend()
ax.set_title("reconstruct_pseudo_open_loop (synthetic WFS input)")
plt.show()

## Summary

Two confirmed findings from this run, both checked analytically/empirically
rather than assumed:

1. `Atmosphere_Characterization.temporal_structure_function`'s `* diameter**2`
   is a scaling bug (section 6) -- exactly 9x too large on this D=3 m file --
   so the old class is not used as a correctness baseline anywhere here, only
   (for the two functions whose math is unchanged) as a check that the
   rewrite reproduces the old formula.
2. `estimate_wind_speed_autocorrelation_cutoff`'s implausible wind speed is
   **not** an open-loop-vs-closed-loop problem (section 9): reconstructing
   pseudo-open-loop input doesn't fix it, confirmed with a controlled
   synthetic experiment where the *true* open-loop atmosphere is *also* wrong
   by the same ~12x factor. The actual cause is that NAOS/Fusco et al. 2004's
   `1.15*pi` constant was calibrated against Conan et al. 1995's exact
   theoretical Zernike PSD shape, and this codebase's simplified
   `_low_pass(f, f_c, 17/3)` atmosphere model only approximates that shape --
   confirmed with a pure, noise-free analytical calculation (~11.7x, matching
   the ~11.8-12x seen on synthetic signals). Don't trust this function's
   absolute output until the constant is recalibrated for the shape actually
   used, or the shape itself is replaced with Conan et al. 1995's exact one.

| function | vs. old formula | notes |
|---|---|---|
| `estimate_r0_L0` | matches, ~4 significant figures | |
| `estimate_tau0_v0_structure_function` | intentionally different, and confirmed correct | old code's `diameter**2` proven wrong in section 6 |
| `estimate_wind_gain_delay_from_psd` | matches, ~4 significant figures (`max_radial_order=6`, `nperseg=n_batch`) | also runs on all 50 modes with a realistic multi-segment `nperseg`, which the original class never does |
| `estimate_wfs_noise_variance` | n/a, new | near-zero here; this file looks noiseless |
| `estimate_wind_speed_autocorrelation_cutoff` | n/a, new | **do not trust the absolute value** -- confirmed ~12x overestimate from a PSD-shape/constant mismatch, independent of loop status (section 9) |
| `detect_secondary_layer_candidate` | n/a, new | no false positive on this single-layer-ish batch |
| `reconstruct_pseudo_open_loop` | n/a, new | mechanically correct (section 11's shape comparison and section 9's controlled experiment both behave as expected); this file has no real WFS time series to fully validate it against |

Two things worth fixing upstream before trusting these on other files:
`DM_commands` needs the per-sample actuator-mean removed before projecting to
Zernike modes (section 2), and this file's `DM_TimeStamps` don't reflect the
true loop rate (also section 2) -- check both assumptions on any other file
before reusing this notebook's approach unmodified.